# logdd · TTS trên Colab

1. Chọn **Runtime → Change runtime type → GPU**.
2. Bấm chạy ô duy nhất bên dưới rồi chọn ZIP tác vụ. Ô này tự chuẩn bị môi trường, tạo audio và tải kết quả.
3. Lưu ZIP kết quả, quay lại TTS Studio → **Nhập kết quả**.

Muốn tải lại file, dùng ô tải kết quả bên dưới. Chạy lại ô chính sẽ tạo lại audio.

Clone dài được chia tự động thành đoạn khoảng 1 phút khi xuất gói, chạy tuần tự với cùng audio mẫu rồi ghép thành một file. Model được giữ trong bộ nhớ cho cả lô. Phiên mới cần tải lại thư viện/model; lần đầu có thể mất vài phút. Không lượng tử hóa model. Giọng thiết kế có thể khác giữa các lần sinh; để giữ một giọng, dùng model Base cùng audio mẫu. Nội dung và audio mẫu được tải lên phiên Colab của bạn.

In [ ]:
def run_logdd_tts():
    from google.colab import files
    from pathlib import Path
    import hashlib, io, json, re, zipfile
    
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one tasks ZIP.')
    data = next(iter(uploaded.values()))
    limit = 256 * 1024 * 1024
    if len(data) > limit:
        raise ValueError('ZIP exceeds 256 MB')
    archive = zipfile.ZipFile(io.BytesIO(data))
    entries = archive.infolist()
    if len(entries) > 1000 or sum(e.file_size for e in entries) > limit:
        raise ValueError('Archive too large')
    names = [e.filename for e in entries]
    if len(names) != len(set(names)):
        raise ValueError('Duplicate archive entries')
    for name in names:
        if not name or name.startswith('/') or '\\' in name or ':' in name or any(p in ('.', '..') for p in name.split('/')):
            raise ValueError('Invalid archive path')
    raw = archive.read('tasks.json')
    manifest = json.loads(raw)
    models = json.loads("{\"cosyvoice3-0.5b\":{\"repository\":\"FunAudioLLM/Fun-CosyVoice3-0.5B-2512\",\"engine\":\"cosyvoice\",\"mode\":\"clone\"},\"qwen3-0.6b-base\":{\"repository\":\"Qwen/Qwen3-TTS-12Hz-0.6B-Base\",\"engine\":\"qwen3\",\"mode\":\"clone\"},\"qwen3-0.6b-custom\":{\"repository\":\"Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice\",\"engine\":\"qwen3\",\"mode\":\"preset\"},\"qwen3-1.7b-base\":{\"repository\":\"Qwen/Qwen3-TTS-12Hz-1.7B-Base\",\"engine\":\"qwen3\",\"mode\":\"clone\"},\"qwen3-1.7b-custom\":{\"repository\":\"Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\",\"engine\":\"qwen3\",\"mode\":\"preset\"},\"qwen3-1.7b-design\":{\"repository\":\"Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign\",\"engine\":\"qwen3\",\"mode\":\"design\"}}")
    model = models.get(manifest.get('modelId'))
    if manifest.get('format') != 'logdd-tts-tasks' or manifest.get('version') != 1 or not model:
        raise ValueError('Unsupported task package')
    if manifest.get('repository') != model['repository'] or manifest.get('engine') != model['engine']:
        raise ValueError('Unsupported model')
    valid_id = lambda value: isinstance(value, str) and re.fullmatch(r'[a-zA-Z0-9_-]{1,100}', value)
    if not valid_id(manifest.get('packId')):
        raise ValueError('Invalid package ID')
    jobs = manifest.get('jobs', [])
    if not isinstance(jobs, list) or not 1 <= len(jobs) <= 100:
        raise ValueError('Expected 1–100 tasks')
    seen = set()
    for job in jobs:
        if not valid_id(job.get('id')) or job['id'] in seen or job.get('mode') != model['mode']:
            raise ValueError('Invalid task')
        seen.add(job['id'])
        parts = job.get('parts')
        if not isinstance(parts, list) or not parts or not all(isinstance(p, str) and p.strip() for p in parts) or sum(map(len, parts)) > 100000:
            raise ValueError('Invalid task text')
        ref = job.get('referenceAudioPath')
        if ref and (ref not in names or not ref.startswith('references/')):
            raise ValueError('Missing reference audio')
    ROOT = Path('/content/logdd-tts') / manifest['packId']
    ROOT.mkdir(parents=True, exist_ok=True)
    (ROOT / 'tasks.json').write_bytes(raw)
    for name in names:
        if name.startswith('references/') and not name.endswith('/'):
            target = (ROOT / name).resolve()
            if not target.is_relative_to(ROOT.resolve()):
                raise ValueError('Invalid reference path')
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(name))
    print(f"Ready: {len(jobs)} tasks · {manifest['repository']}")
    
    # --- NEXT STAGE ---
    
    import subprocess, sys
    from pathlib import Path
    import json, shutil
    if shutil.which('nvidia-smi') is None:
        raise RuntimeError('Chưa có GPU. Chọn Runtime > Change runtime type > GPU rồi chạy lại ô này.')
    subprocess.run(['nvidia-smi'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.12.3'], check=True)
    env = Path('/content/logdd-env-' + manifest['engine'])
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed', '--allow-existing', str(env)], check=True)
    PYTHON = str(env / 'bin/python')
    subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', 'wheel', 'setuptools<81'], check=True)
    if manifest['engine'] == 'qwen3':
        packages = ['torch==2.8.0', 'torchaudio==2.8.0', 'qwen-tts==0.1.1', 'transformers==4.57.3', 'soundfile==0.13.1']
    else:
        packages = ['torch==2.3.1', 'torchaudio==2.3.1', 'numpy==1.26.4', 'transformers==4.51.3', 'conformer==0.3.2', 'diffusers==0.29.0', 'hydra-core==1.3.2', 'HyperPyYAML==1.2.3', 'inflect==7.3.1', 'librosa==0.10.2', 'lightning==2.2.4', 'modelscope==1.20.0', 'onnxruntime==1.18.0', 'rich==13.7.1', 'soundfile==0.12.1', 'x-transformers==2.11.24', 'wetext==0.0.4', 'wget==3.2', 'pyarrow==18.1.0', 'gdown==5.1.0', 'matplotlib==3.9.4', 'Cython==3.0.12']
    subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', *packages], check=True)
    if manifest['engine'] == 'cosyvoice':
        subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'openai-whisper==20231117', 'pyworld==0.3.4'], check=True)
    scripts = json.loads("{\"local_models_worker.py\":\"\\\"\\\"\\\"CosyVoice/Qwen3 adapter with JSON-line serving and cached model/voice prompts.\\\"\\\"\\\"\\nimport contextlib\\nimport io\\nimport json\\nfrom collections import OrderedDict\\nfrom pathlib import Path\\nimport sys\\nimport urllib.request\\nimport zipfile\\n\\nPROTOCOL = sys.stdout\\nCURRENT_JOB = None\\nCACHED_MODEL = None\\nCACHED_KEY = None\\nCLONE_PROMPTS = OrderedDict()\\nCOSY_REVISION = '074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc'\\nMATCHA_REVISION = 'dd9105b34bf2be2230f4aa1e4769fb586a3c824e'\\nLANGUAGES = {'auto': 'Auto', 'zh': 'Chinese', 'en': 'English', 'ja': 'Japanese',\\n             'ko': 'Korean', 'de': 'German', 'fr': 'French', 'ru': 'Russian',\\n             'pt': 'Portuguese', 'es': 'Spanish', 'it': 'Italian'}\\nSPEAKERS = ('Vivian', 'Serena', 'Uncle_Fu', 'Dylan', 'Eric', 'Ryan', 'Aiden', 'Ono_Anna', 'Sohee')\\n\\n\\ndef emit(kind, **values):\\n    print(json.dumps({'type': kind, 'jobId': CURRENT_JOB, **values}, ensure_ascii=False), file=PROTOCOL, flush=True)\\n\\n\\ndef download_source(repository, revision, destination):\\n    destination = Path(destination)\\n    marker = destination / '.revision'\\n    if marker.exists() and marker.read_text() == revision:\\n        return\\n    url = f'https://codeload.github.com/{repository}/zip/{revision}'\\n    with urllib.request.urlopen(url, timeout=120) as response:\\n        archive = zipfile.ZipFile(io.BytesIO(response.read()))\\n    for member in archive.infolist():\\n        relative = Path(*Path(member.filename).parts[1:])\\n        target = (destination / relative).resolve()\\n        if not target.is_relative_to(destination.resolve()):\\n            raise ValueError('Invalid source archive path')\\n        if member.is_dir():\\n            target.mkdir(parents=True, exist_ok=True)\\n        else:\\n            target.parent.mkdir(parents=True, exist_ok=True)\\n            target.write_bytes(archive.read(member))\\n    marker.write_text(revision)\\n\\n\\ndef load_model(request):\\n    if request.get('backend') == 'mlx':\\n        from qwen_mlx_adapter import load_qwen_mlx\\n        return load_qwen_mlx(request), 'mlx'\\n    import torch\\n    # Qwen Base keeps the official SDK's CUDA/CPU path.\\n    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'\\n    if request['engine'] == 'qwen3':\\n        from qwen_tts import Qwen3TTSModel\\n        model = Qwen3TTSModel.from_pretrained(\\n            request['modelPath'], device_map=device,\\n            dtype=torch.bfloat16 if device.startswith('cuda') and torch.cuda.is_bf16_supported() else torch.float32,\\n            attn_implementation='eager',\\n        )\\n    else:\\n        source = Path(request['sourcePath'])\\n        sys.path[:0] = [str(source), str(source / 'third_party' / 'Matcha-TTS')]\\n        from cosyvoice.cli.cosyvoice import AutoModel\\n        model = AutoModel(model_dir=request['modelPath'], load_trt=False, load_vllm=False, fp16=False)\\n        from cosy_mps_adapter import configure_cosy_acceleration\\n        return model, configure_cosy_acceleration(model)\\n    return model, 'cuda' if device.startswith('cuda') else 'cpu'\\n\\n\\ndef get_model(request):\\n    global CACHED_MODEL, CACHED_KEY\\n    key = (request['engine'], request['modelPath'], request.get('backend'))\\n    if key != CACHED_KEY:\\n        CACHED_MODEL = None\\n        CACHED_KEY = None\\n        CLONE_PROMPTS.clear()\\n        CACHED_MODEL = load_model(request)\\n        CACHED_KEY = key\\n    return CACHED_MODEL\\n\\n\\ndef clone_prompt(model, request):\\n    audio = Path(request['referenceAudioPath'])\\n    stat = audio.stat()\\n    transcript = request.get('referenceText', '').strip()\\n    key = (str(audio.resolve()), stat.st_mtime_ns, stat.st_size, transcript)\\n    if key not in CLONE_PROMPTS:\\n        CLONE_PROMPTS[key] = model.create_voice_clone_prompt(\\n            ref_audio=str(audio), ref_text=transcript or None,\\n            x_vector_only_mode=not bool(transcript))\\n        if len(CLONE_PROMPTS) > 8:\\n            CLONE_PROMPTS.popitem(last=False)\\n    CLONE_PROMPTS.move_to_end(key)\\n    return CLONE_PROMPTS[key]\\n\\n\\ndef validate(request):\\n    engine = request.get('engine')\\n    if engine not in ('qwen3', 'cosyvoice'):\\n        raise ValueError('Unsupported TTS engine')\\n    if not request.get('text', '').strip():\\n        raise ValueError('Text is required')\\n    mode = request.get('mode')\\n    if engine == 'cosyvoice':\\n        allowed = 'clone'\\n    else:\\n        variant = request['repository'].rsplit('-', 1)[-1]\\n        allowed = {'Base': 'clone', 'CustomVoice': 'preset', 'VoiceDesign': 'design'}.get(variant)\\n    if mode != allowed:\\n        raise ValueError(f'This model requires {allowed} mode')\\n    if mode == 'clone':\\n        if not request.get('referenceAudioPath') or not Path(request['referenceAudioPath']).is_file():\\n            raise ValueError('Reference audio is required')\\n    if mode == 'design' and not request.get('instruction', '').strip():\\n        raise ValueError('Voice description is required')\\n    if engine == 'qwen3':\\n        if request.get('language', 'auto') not in LANGUAGES:\\n            raise ValueError('Qwen3-TTS does not support this language')\\n        if mode == 'preset' and '-0.6B-' in request['repository'] and request.get('localStyle', '').strip():\\n            raise ValueError('Emotion instructions require Qwen3 CustomVoice 1.7B')\\n        if mode == 'preset' and request.get('qwenSpeaker', 'Ryan') not in SPEAKERS:\\n            raise ValueError('Unknown Qwen3 speaker')\\n\\n\\ndef synthesize(model, request):\\n    if request.get('backend') == 'mlx':\\n        from qwen_mlx_adapter import synthesize_qwen_mlx\\n        return synthesize_qwen_mlx(model, request, LANGUAGES)\\n    style = request.get('localStyle', '').strip()\\n    if request['engine'] == 'qwen3':\\n        kwargs = {'text': request['text'], 'language': LANGUAGES[request.get('language', 'auto')]}\\n        if request['mode'] == 'clone':\\n            return model.generate_voice_clone(**kwargs, voice_clone_prompt=clone_prompt(model, request))\\n        if request['mode'] == 'design':\\n            return model.generate_voice_design(**kwargs, instruct=request['instruction'])\\n        if style:\\n            kwargs['instruct'] = style\\n        return model.generate_custom_voice(**kwargs, speaker=request.get('qwenSpeaker', 'Ryan'))\\n    import torch\\n    prefix = 'You are a helpful assistant.'\\n    reference = request['referenceAudioPath']\\n    transcript = request.get('referenceText', '').strip()\\n    options = {'stream': bool(request.get('streamPreview')), 'text_frontend': False}\\n    if style:\\n        parts = model.inference_instruct2(request['text'], prefix + ' ' + style + '<|endofprompt|>', reference, **options)\\n    elif transcript:\\n        parts = model.inference_zero_shot(request['text'], prefix + '<|endofprompt|>' + transcript, reference, **options)\\n    else:\\n        parts = model.inference_cross_lingual(prefix + '<|endofprompt|>' + request['text'], reference, **options)\\n    chunks = []\\n    for part in parts:\\n        chunk = part['tts_speech'].detach().cpu()\\n        chunks.append(chunk)\\n        if options['stream']:\\n            import soundfile as sf\\n            output = Path(request['outputPath'])\\n            chunk_path = output.with_name(output.stem + f'-chunk-{len(chunks)}.wav')\\n            sf.write(str(chunk_path), chunk.squeeze(0).numpy(), model.sample_rate)\\n            emit('progress', stage='generating', message='Audio ready', audioChunkPath=str(chunk_path))\\n    if not chunks:\\n        raise ValueError('CosyVoice returned no audio')\\n    return [torch.cat(chunks, dim=1).squeeze(0).numpy()], model.sample_rate\\n\\n\\ndef main(request):\\n    if request['command'] == 'prepare':\\n        if request['engine'] == 'cosyvoice':\\n            emit('progress', stage='runtime.dependencies', percent=40, message='Downloading CosyVoice source…')\\n            download_source('QwenAudio/CosyVoice', COSY_REVISION, request['sourcePath'])\\n            download_source('shivammehta25/Matcha-TTS', MATCHA_REVISION,\\n                            Path(request['sourcePath']) / 'third_party' / 'Matcha-TTS')\\n        from huggingface_hub import snapshot_download\\n        emit('progress', stage='model.download', percent=50, message='Downloading model weights…')\\n        snapshot_download(repo_id=request['repository'], local_dir=request['modelPath'])\\n        emit('progress', stage='model.download', percent=90, message='Checking model runtime…')\\n        _, accelerator = load_model(request)\\n        return {'success': True, 'accelerator': accelerator}\\n    if request['command'] != 'generate':\\n        raise ValueError('Unknown worker command')\\n    validate(request)\\n    emit('progress', stage='loading', percent=10, message='Loading model…')\\n    model, accelerator = get_model(request)\\n    emit('progress', stage='generating', percent=30, message='Generating speech…')\\n    waves, sample_rate = synthesize(model, request)\\n    import soundfile as sf\\n    if not waves or len(waves[0]) == 0:\\n        raise ValueError('Model returned no audio')\\n    sf.write(request['outputPath'], waves[0], sample_rate)\\n    return {'success': True, 'sampleRate': sample_rate, 'durationSec': len(waves[0]) / sample_rate,\\n            'accelerator': accelerator}\\n\\n\\ndef handle_request(line):\\n    global CURRENT_JOB\\n    try:\\n        payload = json.loads(line)\\n        CURRENT_JOB = payload.get('jobId')\\n        with contextlib.redirect_stdout(sys.stderr):\\n            result = main(payload)\\n        emit('result', **result)\\n    except Exception as exc:\\n        emit('result', success=False, error=str(exc))\\n    finally:\\n        CURRENT_JOB = None\\n\\n\\nif __name__ == '__main__':\\n    if '--serve' in sys.argv:\\n        for line in sys.stdin:\\n            if line.strip():\\n                handle_request(line)\\n    else:\\n        handle_request(sys.stdin.readline())\\n\",\"cosy_mps_adapter.py\":\"\\\"\\\"\\\"Run CosyVoice3's LLM/flow on Apple GPU, retaining the original CPU vocoder.\\\"\\\"\\\"\\nfrom collections import OrderedDict\\nfrom pathlib import Path\\n\\n\\ndef cache_reference_features(frontend):\\n    # Cache only acoustic features. Text and emotion instructions are still\\n    # encoded on each request, avoiding stale transcripts/style when cloning.\\n    def wrap(extract):\\n        cache = OrderedDict()\\n\\n        def cached(audio):\\n            if not isinstance(audio, (str, Path)):\\n                return extract(audio)\\n            path = Path(audio)\\n            stat = path.stat()\\n            key = (str(path.resolve()), stat.st_mtime_ns, stat.st_size)\\n            if key not in cache:\\n                cache[key] = extract(audio)\\n                if len(cache) > 8:\\n                    cache.popitem(last=False)\\n            cache.move_to_end(key)\\n            result = cache[key]\\n            # Upstream trims the feature/token lengths in place; never let\\n            # those mutations affect a later request using this reference.\\n            return tuple(item.clone() for item in result) if isinstance(result, tuple) else result.clone()\\n\\n        return cached\\n\\n    for name in ('_extract_speech_feat', '_extract_speech_token', '_extract_spk_embedding'):\\n        setattr(frontend, name, wrap(getattr(frontend, name)))\\n\\n\\ndef configure_cosy_acceleration(cosy):\\n    import torch\\n\\n    if torch.cuda.is_available():\\n        return 'cuda'\\n    if not torch.backends.mps.is_available():\\n        return 'cpu'\\n    model = cosy.model\\n    # Keep the checkpoint, FP32 and the original flow step count. HiFT's\\n    # pitch predictor uses float64, which MPS does not support.\\n    model.device = torch.device('mps')\\n    model.llm.to(model.device, dtype=torch.float32)\\n    model.flow.to(model.device, dtype=torch.float32)\\n    model.hift.to('cpu')\\n    hift_inference = model.hift.inference\\n\\n    def cpu_hift(speech_feat, **kwargs):\\n        return hift_inference(speech_feat=speech_feat.to('cpu'), **kwargs)\\n\\n    model.hift.inference = cpu_hift\\n    cache_reference_features(cosy.frontend)\\n    # Upstream generates tokens on a thread. Propagate its errors to the\\n    # caller instead of leaving streaming stuck waiting for more tokens.\\n    errors = {}\\n    llm_job = model.llm_job\\n    token2wav = model.token2wav\\n\\n    def guarded_llm_job(text, prompt_text, speech_token, embedding, uuid):\\n        try:\\n            llm_job(text, prompt_text, speech_token, embedding, uuid)\\n        except Exception as exc:\\n            errors[uuid] = exc\\n        finally:\\n            model.llm_end_dict[uuid] = True\\n\\n    def checked_token2wav(*args, **kwargs):\\n        uuid = kwargs['uuid']\\n        error = errors.pop(uuid, None)\\n        if error is not None:\\n            model.tts_speech_token_dict.pop(uuid, None)\\n            model.llm_end_dict.pop(uuid, None)\\n            model.hift_cache_dict.pop(uuid, None)\\n            raise RuntimeError(f'CosyVoice token generation failed: {error}') from error\\n        return token2wav(*args, **kwargs)\\n\\n    model.llm_job = guarded_llm_job\\n    model.token2wav = checked_token2wav\\n    return 'mps'\\n\",\"colab_runner.py\":\"\\\"\\\"\\\"Trusted Colab batch runner. Task archives contain data, never executable code.\\\"\\\"\\\"\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport sys\\nimport zipfile\\n\\n\\ndef render(root):\\n    import numpy as np\\n    import soundfile as sf\\n    import torch\\n    import local_models_worker as worker\\n    from huggingface_hub import snapshot_download\\n    if not torch.cuda.is_available():\\n        raise RuntimeError('Select Runtime > Change runtime type > GPU, then run again.')\\n    root = Path(root)\\n    raw = (root / 'tasks.json').read_bytes()\\n    manifest = json.loads(raw)\\n    model_path = snapshot_download(repo_id=manifest['repository'])\\n    source = Path('/content/logdd-cosyvoice')\\n    if manifest['engine'] == 'cosyvoice':\\n        worker.download_source('QwenAudio/CosyVoice', worker.COSY_REVISION, source)\\n        worker.download_source('shivammehta25/Matcha-TTS', worker.MATCHA_REVISION, source / 'third_party/Matcha-TTS')\\n    common = dict(engine=manifest['engine'], repository=manifest['repository'],\\n                  modelPath=model_path, sourcePath=str(source), streamPreview=False)\\n    report = dict(format='logdd-tts-results', version=1, packId=manifest['packId'],\\n                  requestHash=hashlib.sha256(raw).hexdigest(), jobs=[])\\n    out = root / 'results'\\n    (out / 'audio').mkdir(parents=True, exist_ok=True)\\n    total_bytes = 0\\n    for index, job in enumerate(manifest['jobs']):\\n        print(f\\\"[{index + 1}/{len(manifest['jobs'])}] {job['name']}\\\", flush=True)\\n        try:\\n            request = {**job, **common}\\n            if job.get('referenceAudioPath'):\\n                reference = (root / job['referenceAudioPath']).resolve()\\n                if not reference.is_relative_to(root.resolve()) or not reference.is_file():\\n                    raise ValueError('Invalid reference audio')\\n                request['referenceAudioPath'] = str(reference)\\n            filename = f\\\"audio/{job['id']}.wav\\\"\\n            temporary = out / f\\\"audio/{job['id']}.partial.wav\\\"\\n            rate, frames = None, 0\\n            writer = None\\n            try:\\n                for part_index, text in enumerate(job['parts']):\\n                    print(f\\\"  Segment {part_index + 1}/{len(job['parts'])}\\\", flush=True)\\n                    request['text'] = text\\n                    worker.validate(request)\\n                    model, accelerator = worker.get_model(request)\\n                    if accelerator != 'cuda':\\n                        raise RuntimeError(f'Expected CUDA, got {accelerator}')\\n                    with torch.inference_mode():\\n                        waves, sample_rate = worker.synthesize(model, request)\\n                    wave = np.asarray(waves[0], dtype=np.float32).reshape(-1)\\n                    if not len(wave) or not np.isfinite(wave).all():\\n                        raise ValueError('Model returned empty or invalid audio')\\n                    if rate is not None and rate != sample_rate:\\n                        raise ValueError('Sample rate changed between segments')\\n                    rate = sample_rate\\n                    gap = round(rate * 0.25) if part_index else 0\\n                    frames += len(wave) + gap\\n                    if total_bytes + frames * 2 + 44 > 240 * 1024 * 1024:\\n                        raise ValueError('Results exceed 240 MB; export a smaller batch')\\n                    if writer is None:\\n                        writer = sf.SoundFile(str(temporary), mode='w', samplerate=rate, channels=1, subtype='PCM_16')\\n                    if gap:\\n                        writer.write(np.zeros(gap, dtype=np.float32))\\n                    writer.write(wave)\\n                    del waves, wave\\n            finally:\\n                if writer is not None:\\n                    writer.close()\\n            if not frames:\\n                raise ValueError('No audio segments')\\n            temporary.replace(out / filename)\\n            data = (out / filename).read_bytes()\\n            total_bytes += len(data)\\n            report['jobs'].append(dict(id=job['id'], status='done', file=filename,\\n                                       sha256=hashlib.sha256(data).hexdigest()))\\n        except Exception as exc:\\n            (out / f\\\"audio/{job['id']}.partial.wav\\\").unlink(missing_ok=True)\\n            print(f'Failed: {exc}', flush=True)\\n            report['jobs'].append(dict(id=job['id'], status='error', error=str(exc)[:2000]))\\n        (out / 'results.json').write_text(json.dumps(report, ensure_ascii=False), encoding='utf-8')\\n    output = root / f\\\"logdd-results-{manifest['packId']}.zip\\\"\\n    with zipfile.ZipFile(output, 'w', compression=zipfile.ZIP_STORED) as archive:\\n        archive.write(out / 'results.json', 'results.json')\\n        for row in report['jobs']:\\n            if row['status'] == 'done':\\n                archive.write(out / row['file'], row['file'])\\n    print(f'Finished: {output}', flush=True)\\n    return output\\n\\n\\nif __name__ == '__main__':\\n    render(sys.argv[1])\\n\"}")
    RUNNER = Path('/content/logdd-runner')
    RUNNER.mkdir(exist_ok=True)
    for name, source in scripts.items():
        (RUNNER / name).write_text(source, encoding='utf-8')
    print('Environment ready.')
    
    # --- NEXT STAGE ---
    
    subprocess.run([PYTHON, str(RUNNER / 'colab_runner.py'), str(ROOT)], check=True)
    report = json.loads((ROOT / 'results/results.json').read_text())
    print('Success:', sum(j['status'] == 'done' for j in report['jobs']), '/', len(report['jobs']))
    for job in report['jobs']:
        if job['status'] == 'error':
            print(job['id'], job['error'])
    
    # --- NEXT STAGE ---
    
    result_zip = ROOT / f"logdd-results-{manifest['packId']}.zip"
    files.download(str(result_zip))

run_logdd_tts()


### Tải lại kết quả (không tạo lại audio)
Ô tùy chọn bên dưới tìm các ZIP đã tạo trong phiên hiện tại. Chọn đúng gói rồi bấm tải. Nếu Colab đã xóa phiên máy, cần chạy lại ô chính và tải ZIP tác vụ lên lần nữa.

In [ ]:
from pathlib import Path
from google.colab import files
import ipywidgets as widgets
from IPython.display import display

def show_logdd_downloads():
    paths = sorted(Path('/content/logdd-tts').glob('*/logdd-results-*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not paths:
        print('Chưa có ZIP kết quả trong phiên này. Hãy chạy ô chính trước.')
        return
    choices = widgets.Dropdown(options=[(p.name, str(p)) for p in paths], description='Gói:')
    button = widgets.Button(description='Tải kết quả')
    button.on_click(lambda _: files.download(choices.value))
    display(choices, button)

show_logdd_downloads()
